In [ ]:
print("hello")

In [ ]:
import os 
import sys
from dotenv import load_dotenv, find_dotenv
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
#from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
#from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
load_dotenv()
GROQ_API_KEY=os.getenv("GROQ_API_key")

In [ ]:
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [ ]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    
)

response = llm.invoke("Explain multicollinearity in simple terms.")
print(response.content)

In [ ]:
def load_pdf_file(data):
    loader = DirectoryLoader(data,
       glob="*.pdf",
       loader_cls=PyPDFLoader, 
       )
    documents = loader.load()
    
    return documents

In [ ]:
data = load_pdf_file(data='data/')

In [ ]:
question_gen=""
for page in data:
    question_gen += page.page_content

In [ ]:
question_gen

In [ ]:
splitter_ques_gen =  RecursiveCharacterTextSplitter(
    chunk_size = 10000,
    chunk_overlap = 200
)

In [ ]:
chunk_ques_gen = splitter_ques_gen.split_text(question_gen)

In [ ]:
from langchain_core.documents import Document

In [ ]:
document_ques_gen = [Document(page_content= t) for t in chunk_ques_gen]

In [ ]:
embeding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
vectorsstore = FAISS.from_documents(document_ques_gen, embeding)

In [ ]:
retrive = vectorsstore.as_retriever()

In [ ]:
prompt = ChatPromptTemplate.from_template(
    "Generate three insightful questions based on the following context:\n{context}\nQuestions:"
)

In [ ]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    {"context": retrive, "question": RunnablePassthrough(), "answer": RunnablePassthrough()}
    | prompt_template
    | llm
)

chain.invoke("make 5 questions ")

In [ ]:
response = chain.invoke("make 5 questions")